In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
root_loc = '/Volumes/workspace/default/yahoo_data'

schema = StructType([
    StructField("Symbol", StringType(), False),
    StructField("Current Price", DoubleType(), True),
    StructField("Date",StringType(), True),
    StructField("Time", StringType(), True),
    StructField("Change", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Volume", DoubleType(), True),
    StructField("Trade Date",StringType(), True),
    StructField("Purchase Price", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),
    StructField("High Limit", DoubleType(), True),
    StructField("Low Limit", DoubleType(), True),
    StructField("Comment", StringType(), True),
    StructField("Transaction Type", StringType(), True)
    ])

In [0]:
df = (
  spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(schema)
    .load(root_loc)
)
df_clean = df.selectExpr(
    "Symbol",
    "`Current Price` as Current_Price",
    "Date",
    "Time",
    "Change",
    "Open",
    "High",
    "Low",
    "Volume",
    "`Trade Date` as Trade_Date",
    "`Purchase Price` as Purchase_Price",
    "Quantity",
    "`High Limit` as High_Limit",
    "`Low Limit` as Low_Limit",
    "Comment",
    "`Transaction Type` as Transaction_Type",
    '_rescued_data'
)

In [0]:
(
  df_clean.writeStream
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/yahoo_data_checkpoint")
    .trigger(availableNow=True)
    .toTable("workspace.default.yahoo_raw_data")
)